## Retrieval-Augmented Generation (RAG) service (1 point)

In [1]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("./milvus_rag.db")

In [2]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")

In [3]:
COLLECTION_NAME = "rag_texts_and_embeddings"

# re-running the notebook: drop the old collection, otherwise create_collection fails (and insert would duplicate rows)
if milvus_client.has_collection(COLLECTION_NAME):
    milvus_client.drop_collection(COLLECTION_NAME)

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding", 
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
) 

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))

['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 0, 'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 0, 'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True}


In [4]:
# define data source and destination
## the document origin destination from which document will be downloaded 
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document 
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"

In [5]:
# download data
import os
import requests

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)

In [6]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)

extract_pdf_text(file_name, file_json)

In [7]:
# vectorize data

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)

In [8]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r") as t_f, open(os.path.join(data_dir, embeddings_json), "r") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))
        
        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")

In [9]:
# search

def search(model, query, client=milvus_client):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings", 
        data=[embedded_query], 
        limit=1,
        search_params={"metric_type": "L2"},
        output_fields=["text"]
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

In [10]:
result

data: [[{'id': 7, 'distance': 29.125173568725586, 'entity': {'text': 'Historia powstania\nsztucznej inteligencji\n7\nW języku potocznym „sztuczny" oznacza to, co\njest \nwytworem \nmającym \nnaśladować \ncoś\nnaturalnego. W takim znaczeniu używamy\nterminu ,,sztuczny\'\', gdy mówimy o sztucznym\nlodowisku lub oku. Sztuczna inteligencja byłaby\nczymś (programem, maszyną) symulującym\ninteligencję naturalną, ludzką.\nSztuczna inteligencja (AI) to obszar informatyki,\nktóry skupia się na tworzeniu programów\nkomputerowych zdolnych do wykonywania\nzadań, które wymagają ludzkiej inteligencji. \nTe zadania obejmują rozpoznawanie wzorców,\nrozumienie języka naturalnego, podejmowanie\ndecyzji, uczenie się, planowanie i wiele innych.\nGłównym celem AI jest stworzenie systemów,\nktóre są zdolne do myślenia i podejmowania\ndecyzji na sposób przypominający ludzki.\nHistoria sztucznej inteligencji sięga lat 50. \nXX wieku, kiedy to powstały pierwsze koncepcje\ni modele tego, co mogłoby stać się szt

In [11]:
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
import os
from google import genai

GEMINI_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_KEY)

MODEL = "gemini-3.8-flash"

def generate_response(prompt: str):
    try:
        # Send request to Gemini 2.5 Flash API and get the response
        response = gemini_client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        # join the text parts ourselves - response.text warns about 'thought_signature' parts
        return "".join(part.text for part in response.candidates[0].content.parts if part.text)
    except Exception as e:
        print(f"Error generating response: {e}")
        return None

In [13]:
def build_prompt(context: str, query: str, instruction: str = "") -> str:
    prompt = f"""Odpowiedz na pytanie, korzystając wyłącznie z poniższego kontekstu.
    Jeśli w kontekście nie ma informacji potrzebnych do odpowiedzi, napisz: "Nie znalazłem tej informacji w źródłach."
    Nie wymyślaj faktów spoza kontekstu.
    {instruction}

    Kontekst:
    {context}

    Pytanie: {query}

    Odpowiedź:"""
    return prompt


def rag(model, query: str, instruction: str = "") -> str:
    result = search(model, query)
    context = "\n\n".join(hit["entity"]["text"] for hit in result[0])
    prompt = build_prompt(context, query, instruction)
    return generate_response(prompt)

### Testing the RAG system

In [14]:
print(rag(model, "Czym jest sztuczna inteligencja"))
print()
print(rag(model, "Jakie są zastosowania AI w marketingu?"))

Na podstawie tekstu sztuczna inteligencja (AI) to:

* **obszar informatyki**, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji (takich jak rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się czy planowanie);
* **program, maszyna lub system** symulujący inteligencję naturalną (ludzką) i realizujący niektóre funkcje umysłu – czasami w sposób przewyższający funkcje naturalne (np. będąc wolnym od defektów pamięci czy pomyłek przy liczeniu).

Na podstawie tekstu zastosowania AI w marketingu obejmują:

* **Analizę danych** – szybkie przetwarzanie dużych ilości danych za pomocą modeli predykcyjnych w celu identyfikacji trendów i wzorców, co pomaga w tworzeniu skuteczniejszych strategii.
* **Agregowanie i przewidywanie trendów** – prognozowanie przyszłych trendów na bazie danych historycznych, co ułatwia podejmowanie decyzji i planowanie kampanii marketingowych.
* **Monitorowanie konkurenc

#### Style

In [15]:
print("Pirat:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiadaj jak pirat."))
print()

print("Szekspir:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiadaj językiem Szekspira (archaiczny, wzniosły styl)."))
print()

print("Yoda:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiadaj jak mistrz Yoda (nietypowy szyk zdania)."))
print()

Pirat:
Ahoj, kamracie! Z tych morskich zapisków wynika, że sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań, które normalnie wymagają ludzkiego rozumu, do stu beczek rumu! 

Mowa tu o programie lub maszynie, co to symuluje naturalną, ludzką inteligencję. Jej celem jest sprawić, by machiny myślały i podejmowały decyzje na sposób przypominający człowieka – na przykład uczyły się, planowały, podejmowały decyzje, rozumiały ludzki język czy rozpoznawały wzorce, arrr!

Szekspir:
Oto, czym jest owo zjawisko, co je sztuczną inteligencją mianują: toć obszar kunsztu informatycznego, który zamysł swój kładzie na powoływaniu do życia programów komputerowych, zdatnych do sprawowania dzieł wymagających człeczego rozumu. Jest to twór – program aliści maszyna – co inteligencję naturalną, ludzką symulować pragnie, by myśleć oraz wyroki swe podejmować na kształt i podobieństwo ludzkie. 

Rzeczone dzieła, do których jest

#### Language

In [16]:
print("Angielski:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiedz po angielsku."))
print()

print("Łacina:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiedz po łacinie."))
print()

print("Japoński:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiedz po japońsku."))
print()

Angielski:
Based on the provided text, artificial intelligence (AI) is an area of computer science that focuses on creating computer programs capable of performing tasks that require human intelligence, such as pattern recognition, understanding natural language, decision-making, learning, planning, and many others. 

In a general sense, it is something (a program, a machine, or a system) that simulates natural, human intelligence and executes certain functions of the mind (sometimes even surpassing natural functions, for example by being free from calculation errors or memory defects). Its main goal is to create systems capable of thinking and making decisions in a way that resembles humans.

Łacina:
Secundum contextum, intelligentia artificialis est campus informaticae, qui spectat ad programmata computatralia creanda apta ad opera perficienda quae intelligentiam humanam requirunt. Significat etiam aliquid (programma vel machinam) quod intelligentiam naturalem et humanam simulat.

Ja

#### Format

In [17]:
print("Haiku:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiedz w formie Naukowca"))
print()

print("Jedno zdanie:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiedz jednym zdaniem."))
print()

print("Trzy punkty:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiedz dokładnie trzema punktami (bullet points)."))
print()

print("Emoji:")
print(rag(model, "Czym jest sztuczna inteligencja", "Dla każdego faktu podaj emoji i jednolinijkowe wyjaśnienie."))
print()

Haiku:
W świetle przedstawionych danych źródłowych, sztuczną inteligencję (AI) należy zdefiniować wieloaspektowo:

1. **Jako dyscyplinę badawczą:** Jest to obszar informatyki, który koncentruje się na tworzeniu programów komputerowych zdolnych do realizacji zadań wymagających ludzkiej inteligencji (takich jak uczenie się, planowanie, podejmowanie decyzji, rozumienie języka naturalnego czy rozpoznawanie wzorców). Głównym celem AI jest opracowanie systemów zdolnych do myślenia i decydowania w sposób przypominający ludzki.

2. **W ujęciu ontologicznym i operacyjnym:** Jest to byt techniczny (program bądź maszyna) symulujący inteligencję naturalną, czyli ludzką. Mamy tu na uwadze system realizujący wybrane funkcje umysłu – niekiedy w sposób przewyższający ich naturalne odpowiedniki (np. poprzez eliminację defektów pamięci czy pomyłek rachunkowych).

Jedno zdanie:
Sztuczna inteligencja (AI) to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania 

#### Persona

In [18]:
print("Profesor:")
print(rag(model, "Czym jest sztuczna inteligencja", "Wciel się w zrzędliwego profesora."))
print()

print("Youtuber:")
print(rag(model, "Czym jest sztuczna inteligencja", "Wciel się w nadmiernie entuzjastycznego youtubera."))
print()

print("Recenzent:")
print(rag(model, "Czym jest sztuczna inteligencja", "Wciel się w sceptycznego recenzenta: oceń każde twierdzenie jako poparte lub niepoparte kontekstem."))
print()

Profesor:
Ech, znowu muszę marnować czas i tłumaczyć rzeczy absolutnie elementarne, bo szanowny student nie potrafi sam złożyć kilku zdań z tekstu! Proszę łaskawie wytężyć uwagę i zanotować, bo powtarzać nie będę:

Według przedstawionych materiałów, sztuczna inteligencja (AI) to **obszar informatyki**, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji – takich jak rozpoznawanie wzorców, rozumienie języka naturalnego, podejmowanie decyzji, uczenie się czy planowanie. 

Mówiąc prościej: to program lub maszyna symulująca inteligencję naturalną (ludzką). Chodzi o system, który ma realizować niektóre funkcje umysłu – zdolny do myślenia i podejmowania decyzji na sposób przypominający ludzki, a czasami nawet przewyższający funkcje naturalne (na przykład poprzez brak pomyłek przy liczeniu czy defektów pamięci).

Czy to wreszcie jasne, czy mam to przeliterować?!

Youtuber:
SIEMANKO WSZYSTKIM! ZOSTAWCIĘ ŁAPKĘ W GÓRĘ I WALNIJCIE

#### Constraints

In [19]:
print("Max 50 słów:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiedz maksymalnie w 50 słowach."))
print()

print("Bez litery e:")
print(rag(model, "Czym jest sztuczna inteligencja", "Odpowiedz bez użycia litery e."))
print()

print("Kolejne litery:")
print(rag(model, "Czym jest sztuczna inteligencja", "Każde zdanie zacznij od kolejnej litery alfabetu."))
print()

Max 50 słów:
Sztuczna inteligencja (AI) to obszar informatyki skupiający się na tworzeniu programów komputerowych zdolnych do wykonywania zadań wymagających ludzkiej inteligencji. Jest czymś (programem, maszyną) symulującym inteligencję naturalną, ludzką, zdolnym do myślenia i podejmowania decyzji na sposób przypominający ludzki.

Bez litery e:
To obszar informatyki skupiony na programach zdolnych do wykonywania zadań.

Kolejne litery:
**A**I to obszar informatyki, który skupia się na tworzeniu programów komputerowych zdolnych do wykonywania zadań, które wymagają ludzkiej inteligencji. 

**B**yłaby ona czymś (programem, maszyną) symulującym inteligencję naturalną, ludzką. 

**C**elem głównym AI jest stworzenie systemów, które są zdolne do myślenia i podejmowania decyzji na sposób przypominający ludzki.



#### Hallucination probe

In [20]:
print("Mistrzostwa 2018:")
print(rag(model, "Kto wygrał Mistrzostwa Świata w piłce nożnej w 2018 roku?"))
print()

print("Bigos:")
print(rag(model, "Jaki jest przepis na bigos?"))
print()

print("PKB:")
print(rag(model, "Ile wynosi PKB Polski w 2023 roku?"))

Mistrzostwa 2018:
Nie znalazłem tej informacji w źródłach.

Bigos:
Nie znalazłem tej informacji w źródłach.

PKB:
Nie znalazłem tej informacji w źródłach.
